# Phase 0 — LAUSD 2024-25 manual checks
Verify by hand that SACS and CAASPP identify LAUSD with matching values **before** trusting any automated join (README, Phase 0).

Findings when first run (2026-09-28):
- SACS `LEAs`: Ccode `19`, Dcode `64733`, "Los Angeles Unified", K-12 ADA 336,446.66
- CAASPP 2025 district row: County `19`, District `64733`, School `0000000`. Both use native CDS parts, so no crosswalk is needed.
- The Decimal decoder reproduces the state Fund 01 total in `UserGL_Totals` to the cent ($115,501,056,408.93).
- Charter-coded rows in LAUSD Fund 01 are balance-sheet only (objects 9xxx, net $0). **Correction (2026-10-02):** that does *not* make the spending district-only. 51 affiliated charters report inside LAUSD's Fund 01 without a school code. Their ADA (34,895 in 2024-25, Form A line C4 / SACS `Charters` with FundUsed = General) is excluded from `LEAs.K12ADA`, so the pipeline's denominator is now K-12 ADA + Fund 01 charter ADA = 371,342.10, the same as CDE's Current Expense ADA.

In [1]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from cds_lookup import CDS, caaspp_entity
import sacs_parser, caaspp_parser
cds = CDS.parse('19647330000000')

## 1. Identifiers in each source

In [2]:
sacs_parser.lea(cds)

{'district_name': 'Los Angeles Unified',
 'district_type': 'Unified School District',
 'k12_ada': 336446.66,
 'fund01_charter_ada': 34895.44,
 'ada': 371342.1}

In [3]:
caaspp_entity(cds, '2025')

{'County Code': '19',
 'District Code': '64733',
 'School Code': '0000000',
 'Type ID': '6',
 'Filler': '',
 'Test Year': '2025',
 'County Name': 'Los Angeles',
 'District Name': 'Los Angeles Unified',
 'School Name': '',
 'Zip Code': ''}

## 2. Raw CAASPP district rows (all students, every grade)

In [4]:
df = pd.read_csv('../data/raw/caaspp/sb_ca2025_1_csv_v1.txt', sep='^', dtype=str, keep_default_na=False, encoding='latin-1')
lausd = df[(df['County Code']=='19') & (df['District Code']=='64733') & (df['School Code']=='0000000') & (df['Student Group ID']=='1')]
lausd[['Test ID','Grade','Total Students Tested with Scores','Percentage Standard Met and Above']]

,Test ID,Grade,Total Students Tested with Scores,Percentage Standard Met and Above
24876,1,3,31173,43.61
24877,1,4,31168,44.99
24878,1,5,32005,47.18
24879,1,6,26858,45.74
24880,1,7,27348,45.89
24881,1,8,28085,44.91
24882,1,11,26807,53.51
24883,1,13,203444,46.45
24884,2,3,31588,47.62
24885,2,4,31591,43.30


## 3. Raw SACS General Fund by object group (district vs charter-coded rows)

In [5]:
gl = sacs_parser.general_ledger(cds, '2425')
f1 = gl[gl.Fund=='01'].assign(charter=lambda d: d.SchoolCode!='0000000', objgrp=lambda d: d.Object.str[0]+'000s', v=lambda d: d.Value.map(float))
f1.pivot_table(index='objgrp', columns='charter', values='v', aggfunc='sum').map(lambda x: f'${x/1e6:,.1f}M')

charter,False,True
objgrp,,
1000s,"$4,087.9M",$nanM
2000s,"$1,619.4M",$nanM
3000s,"$3,154.4M",$nanM
4000s,$460.5M,$nanM
5000s,"$1,675.2M",$nanM
6000s,$146.1M,$nanM
7000s,$644.4M,$nanM
8000s,"$9,823.9M",$nanM
9000s,"$22,354.0M",$0.0M


## 4. Phase 1 output

In [6]:
import join
join.build(cds, ['2024-25'])

,district_name,year,function,spend_per_pupil,ela_pct_met_or_exceeded,math_pct_met_or_exceeded,spend_total,ada,k12_ada,fund01_charter_ada,data_status
0,Los Angeles Unified,2024-25,1000 Instruction,15233.19,46.45,36.76,5.656726e+09,371342.1,336446.66,34895.44,both
1,Los Angeles Unified,2024-25,2000 Instruction-Related Services,4836.08,46.45,36.76,1.795839e+09,371342.1,336446.66,34895.44,both
2,Los Angeles Unified,2024-25,3000 Pupil Services,3724.34,46.45,36.76,1.383005e+09,371342.1,336446.66,34895.44,both
3,Los Angeles Unified,2024-25,4000 Ancillary Services,703.05,46.45,36.76,2.610720e+08,371342.1,336446.66,34895.44,both
4,Los Angeles Unified,2024-25,5000 Community Services,16.22,46.45,36.76,6.023322e+06,371342.1,336446.66,34895.44,both
5,Los Angeles Unified,2024-25,7000 General Administration,1951.33,46.45,36.76,7.246101e+08,371342.1,336446.66,34895.44,both
6,Los Angeles Unified,2024-25,8000 Plant Services,3043.27,46.45,36.76,1.130096e+09,371342.1,336446.66,34895.44,both
7,Los Angeles Unified,2024-25,9000 Other Outgo,2.31,46.45,36.76,8.593895e+05,371342.1,336446.66,34895.44,both


## 5. Spot checks against public sites (checked 2026-09-29)
Three schools, one per growth band and each a different span. The published values were read in Chrome from:
- the CAASPP results site (caaspp-elpac.ets.org, "Test Results at a Glance", all students, all grades)
- the California School Dashboard (enrollment)
- DataQuest ("Free and Reduced Price Meal – School summary data", 2024-25)

Differences of 0.01 are rounding: the CAASPP site adds the rounded "Met" and "Exceeded" percentages (e.g. 25.41 + 21.05 = 46.46), while the research file computes the total from counts.

**School per-pupil spending (ESSA) has no public CDE web page.** It is published only as the data file we use, plus a login-only app for districts. SARC spending figures are prior-year and use a different definition. Spending is checked by reconciliation instead: the ESSA school total is $10,685M vs the SACS General Fund's $10,958M (2.5%).

In [7]:
published = pd.DataFrame([
    # cds, measure, published value
    ('19647336016612', 'ela_pct_met', 36.04), ('19647336016612', 'math_pct_met', 39.13),
    ('19647336016612', 'ela_pct_met_2024', 37.61), ('19647336016612', 'math_pct_met_2024', 45.95),
    ('19647336016612', 'enrollment', 265), ('19647336016612', 'pct_frpm', 95.8),
    ('19647336058200', 'ela_pct_met', 33.76), ('19647336058200', 'math_pct_met', 15.26),
    ('19647336058200', 'ela_pct_met_2024', 26.91), ('19647336058200', 'math_pct_met_2024', 12.73),
    ('19647336058200', 'enrollment', 670), ('19647336058200', 'pct_frpm', 94.8),
    ('19647330126557', 'ela_pct_met', 52.00), ('19647330126557', 'math_pct_met', 14.28),
    ('19647330126557', 'ela_pct_met_2024', 38.89), ('19647330126557', 'math_pct_met_2024', 8.89),
    ('19647330126557', 'enrollment', 411), ('19647330126557', 'pct_frpm', 90.0),
], columns=['cds', 'measure', 'published'])
import school_data
schools, _ = school_data.build(cds)
ours = schools.set_index('cds')
published['ours'] = [ours.loc[c, m] for c, m in zip(published.cds, published.measure)]
published['school'] = published.cds.map(ours['school_name'])
tol = published.measure.map(lambda m: 0.06 if m == 'pct_frpm' else 0.011)  # FRPM published to 1 decimal
published['match'] = (published.ours - published.published).abs() <= tol
published[['school', 'measure', 'ours', 'published', 'match']].round(2)

,school,measure,ours,published,match
0,Compton Avenue Elementary,ela_pct_met,36.04,36.04,True
1,Compton Avenue Elementary,math_pct_met,39.13,39.13,True
2,Compton Avenue Elementary,ela_pct_met_2024,37.61,37.61,True
3,Compton Avenue Elementary,math_pct_met_2024,45.95,45.95,True
4,Compton Avenue Elementary,enrollment,265.00,265.00,True
5,Compton Avenue Elementary,pct_frpm,95.85,95.80,True
6,Northridge Middle,ela_pct_met,33.76,33.76,True
7,Northridge Middle,math_pct_met,15.26,15.26,True
8,Northridge Middle,ela_pct_met_2024,26.92,26.91,True
9,Northridge Middle,math_pct_met_2024,12.73,12.73,True


In [8]:
# District level (CAASPP site): 2025 ELA 46.46 / math 36.76, 2024 ELA 43.06 / math 32.83
caaspp_parser.proficiency(cds, '2025'), caaspp_parser.proficiency(cds, '2024')

({'ela_pct_met_or_exceeded': 46.45, 'math_pct_met_or_exceeded': 36.76},
 {'ela_pct_met_or_exceeded': 43.06, 'math_pct_met_or_exceeded': 32.83})

## 6. Multi-year (PHASE1_TASKS.md): 2022-23 → 2024-25
- **Ground truth:** `data/ground_truth/lausd_expected.csv`, built by `src/ground_truth.py` from LAUSD's board-approved Form 01 filings (CDE SACS Data Viewer) and from the CAASPP site. Tested by `tests/test_lausd_ground_truth.py`.
- **Identifiers:** see `notes/task2_identifiers.md`. The schema differences between years are the SACS `Dname` width and the CAASPP 2023 column order.

In [9]:
import join
table = join.build(cds, join.year_range('2022-23:2024-25'))
table.groupby('year').agg(spend_total=('spend_total','sum'), k12_ada=('k12_ada','first'),
                          spend_per_pupil=('spend_per_pupil','sum'),
                          ela=('ela_pct_met_or_exceeded','first'), math=('math_pct_met_or_exceeded','first'),
                          status=('data_status','first'))

,spend_total,k12_ada,spend_per_pupil,ela,math,status
year,,,,,,
2022-23,9.290098e+09,348194.24,24214.42,41.17,30.50,both
2023-24,1.038529e+10,345338.43,27263.35,43.06,32.83,both
2024-25,1.095823e+10,336446.66,29509.79,46.45,36.76,both


In [10]:
expected = pd.read_csv('../data/ground_truth/lausd_expected.csv', keep_default_na=False)
rows = []
for y in sorted(expected[expected.measure=='gf_expenditure'].year.unique()):
    ours = sacs_parser.form01_function_totals(cds, join.year_codes(y)[0])
    e = expected[(expected.year==y)&(expected.measure=='gf_expenditure')].set_index('function')['expected']
    rows += [{'year': y, 'function': f, 'filing': v, 'ours': float(ours.get(f, 0)), 'diff': float(ours.get(f, 0)) - v} for f, v in e.items()]
pd.DataFrame(rows)

,year,function,filing,ours,diff
0,2022-23,1000 Instruction,5.101458e+09,5.101458e+09,0.0
1,2022-23,2000 Instruction-Related Services,1.605635e+09,1.605635e+09,0.0
2,2022-23,3000 Pupil Services,8.855098e+08,8.855098e+08,0.0
3,2022-23,4000 Ancillary Services,1.697654e+08,1.697654e+08,0.0
4,2022-23,5000 Community Services,4.323917e+06,4.323917e+06,0.0
5,2022-23,6000 Enterprise,-1.936300e+02,-1.936300e+02,0.0
6,2022-23,7000 General Administration,6.009592e+08,6.009592e+08,0.0
7,2022-23,8000 Plant Services,1.025999e+09,1.025999e+09,0.0
8,2022-23,9000 Other Outgo,2.868488e+07,2.868488e+07,0.0
9,2022-23,TOTAL,9.422335e+09,9.422335e+09,0.0
